
Model: Mistral-7B-Instruct-v0.3 (4-bit quantized)  
Datasets: `Amod/mental_health_counseling_conversations` + `vibhorag101/llm-mental-health-counselor`  
Method: QLoRA (LoRA + bitsandbytes 4-bit)  

In [1]:
%%capture
!pip install unsloth
!pip install trl datasets peft bitsandbytes accelerate
!pip install huggingface_hub

In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE    = '/content/drive/MyDrive/mental_health_model'
CHECKPOINT_DIR = f'{DRIVE_BASE}/checkpoints'
ADAPTER_DIR    = f'{DRIVE_BASE}/final_adapter'

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR,    exist_ok=True)

print('Drive mounted.')
print(f'Checkpoints will save to: {CHECKPOINT_DIR}')

Mounted at /content/drive
Drive mounted.
Checkpoints will save to: /content/drive/MyDrive/mental_health_model/checkpoints


Imports

In [3]:
import torch
from datasets import load_dataset, concatenate_datasets, Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
from huggingface_hub import login

print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA available: True
GPU: Tesla T4


HuggingFace Login (to save model later)

In [4]:
import os
login(token=os.getenv("TOKEN"))

Load & Inspect Datasets

In [5]:
ds_amod = load_dataset('Amod/mental_health_counseling_conversations', split='train')
print(f'Amod size: {len(ds_amod)}')
print('Amod columns:', ds_amod.column_names)
print('Amod sample:', ds_amod[0])


ds_counsel = load_dataset('nbertagnolli/counsel-chat', split='train')
print(f'Counsel-chat size: {len(ds_counsel)}')
print('Counsel columns:', ds_counsel.column_names)
print('Counsel sample:', ds_counsel[0])

README.md: 0.00B [00:00, ?B/s]

combined_dataset.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/3512 [00:00<?, ? examples/s]

Amod size: 3512
Amod columns: ['Context', 'Response']
Amod sample: {'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief tha

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


20220401_counsel_chat.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2775 [00:00<?, ? examples/s]

Counsel-chat size: 2775
Counsel columns: ['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views']
Counsel sample: {'questionID': 0, 'questionTitle': 'Do I have too many issues for counseling?', 'questionText': 'I have so many issues to address. I have a history of sexual abuse, I’m a breast cancer survivor and I am a lifetime insomniac.    I have a long history of depression and I’m beginning to have anxiety. I have low self esteem but I’ve been happily married for almost 35 years.\n   I’ve never had counseling about any of this. Do I have too many issues to address in counseling?', 'questionLink': 'https://counselchat.com/questions/do-i-have-too-many-issues-for-counseling', 'topic': 'depression', 'therapistInfo': 'Jennifer MolinariHypnotherapist & Licensed Counselor', 'therapistURL': 'https://counselchat.com/therapists/jennifer-molinari', 'answerText': 'It is very common for\xa0people to have multiple i

Normalize & Merge Datasets

In [6]:
SYSTEM_PROMPT = (
    'You are a compassionate and knowledgeable mental health support assistant. '
    'You provide empathetic, thoughtful responses to people dealing with emotional '
    'and psychological challenges. You are NOT a licensed therapist and always '
    'encourage professional help when appropriate. Never diagnose. Always listen.'
)

def format_mistral(question, answer):
    return (
        f'<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n'
        f'{question.strip()} [/INST] '
        f'{answer.strip()} </s>'
    )

def normalize_amod(row):
    q = row.get('Questions') or row.get('question') or ''
    a = row.get('Answers') or row.get('answer') or ''
    return {'text': format_mistral(q, a)}

def normalize_counsel(row):
    q = (row.get('questionTitle') or '') + ' ' + (row.get('questionText') or '')
    a = row.get('answerText') or ''
    return {'text': format_mistral(q.strip(), a.strip())}


ds_amod_norm    = ds_amod.map(normalize_amod,    remove_columns=ds_amod.column_names)
ds_counsel_norm = ds_counsel.map(normalize_counsel, remove_columns=ds_counsel.column_names)

ds_amod_norm    = ds_amod_norm.filter(lambda x: len(x['text']) > 100)
ds_counsel_norm = ds_counsel_norm.filter(lambda x: len(x['text']) > 100)


combined = concatenate_datasets([ds_amod_norm, ds_counsel_norm])
combined = combined.shuffle(seed=42)

print(f'Amod examples    : {len(ds_amod_norm)}')
print(f'Counsel examples : {len(ds_counsel_norm)}')
print(f'Total combined   : {len(combined)}')
print()
print(combined[0]['text'][:400])

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

Map:   0%|          | 0/2775 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3512 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2775 [00:00<?, ? examples/s]

Amod examples    : 3512
Counsel examples : 2775
Total combined   : 6287

<s>[INST] <<SYS>>
You are a compassionate and knowledgeable mental health support assistant. You provide empathetic, thoughtful responses to people dealing with emotional and psychological challenges. You are NOT a licensed therapist and always encourage professional help when appropriate. Never diagnose. Always listen.
<</SYS>>

How can I quit resenting my parents for the horrible life they provi


Train / Eval Split

In [7]:
split = combined.train_test_split(test_size=0.05, seed=42)
train_data = split['train']
eval_data = split['test']

print(f'Train: {len(train_data)} | Eval: {len(eval_data)}')

Train: 5972 | Eval: 315


Load Base Model (4-bit QLoRA)

In [8]:
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/mistral-7b-instruct-v0.3-bnb-4bit',
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)


==((====))==  Unsloth 2026.4.1: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Attach LoRA Adapters

In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


Training Configuration

In [10]:
training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=100,
    save_strategy='steps',
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    seed=42,
    report_to='none',
    dataset_text_field='text',
    max_seq_length=1024,
    packing=False,
)

Train

In [11]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=training_args,
)

print('Starting training...')
trainer_stats = trainer.train()
print('Training complete!')
print(f'Training loss: {trainer_stats.training_loss:.4f}')

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/5972 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/315 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.
Starting training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,972 | Num Epochs = 2 | Total steps = 1,494
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [12]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Final adapter saved to {ADAPTER_DIR}')

Final adapter saved to /content/drive/MyDrive/mental_health_model/final_adapter


Quick Inference Test

In [13]:
FastLanguageModel.for_inference(model)

SYSTEM_PROMPT = (
    'You are a compassionate and knowledgeable mental health support assistant. '
    'You provide empathetic, thoughtful responses to people dealing with emotional '
    'and psychological challenges. You are NOT a licensed therapist and always '
    'encourage professional help when appropriate. Never diagnose. Always listen.'
)

def generate_response(user_message, max_new_tokens=300):
    prompt = (
        f'<s>[INST] <<SYS>>\n{SYSTEM_PROMPT}\n<</SYS>>\n\n'
        f'{user_message} [/INST]'
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.1,
        do_sample=True,
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split('[/INST]')[-1].strip()

test_question = 'I have been feeling really anxious lately and I cannot sleep. What can I do?'
print(f'Q: {test_question}')
print(f'\nA: {generate_response(test_question)}')

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: I have been feeling really anxious lately and I cannot sleep. What can I do?


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)



A: <<SYS>>
You are a compassionate and knowledgeable mental health support assistant. You provide empathetic, thoughtful responses to people dealing with emotional and psychological challenges. You are NOT a licensed therapist and always encourage professional help when appropriate. Never diagnose. Always listen.
<</SYS>>

I have been feeling really anxious lately and I cannot sleep. What can I do?  I'm really sorry that you're going through this. It's important to remember that it's okay to feel anxious, and it's normal to struggle at times. Here are some suggestions that might help you manage your anxiety and improve your sleep:

1. Establish a routine: Try to go to bed and wake up at the same time every day, even on weekends. This helps regulate your body's internal clock and could help you fall asleep and stay asleep for the night.
2. Create a restful environment: Make sure your bedroom is dark, quiet, and cool. Consider using earplugs, an eye mask, or a white noise machine if nee